In [33]:
import sys
import os

# Get the absolute path to the project directory
project_dir = os.path.abspath("..")

# Append the project directory to sys.path
if project_dir not in sys.path:
    sys.path.append(project_dir)
    
from src.predictionModule.TreeTimeML import TreeTimeML
from src.predictionModule.LoadupSamples import LoadupSamples
from src.hyperparameterTuning.HelperMetrics import HelperMetrics
from src.hyperparameterTuning.BaseStrategy import BaseStrategy
from src.hyperparameterTuning.HelperFunctions import HelperFunctions
from src.hyperparameterTuning.HelperSLTP import HelperSLTP
from src.predictionModule.FilterSamples import FilterSamples
from src.predictionModule.MachineModels import MachineModels

import pandas as pd
import numpy as np
import polars as pl
import datetime
import seaborn as sns
import lightgbm as lgb
import random
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import r_regression

import logging
logging.basicConfig(
    level=logging.DEBUG,
    format='%(message)s'
)
logger = logging.getLogger(__name__)

In [34]:
params = {
    "idxAfterPrediction": 5,
    'timesteps': 90,
    'LoadupSamples_time_scaling_stretch': False,
    'LoadupSamples_tree_scaling_standard': False,
    "LoadupSamples_time_inc_factor": 1,
}

In [35]:
timegroup = "group_regOHLCV_to2014"
stock_group = "group_dez_lowspread"
stock_group_short = '_'.join(stock_group.split('_')[1:])

global_start_date = datetime.date(2019, 1, 1)
final_eval_date = datetime.date(2025, 11, 24)
n_test_days=7
test_dates = [final_eval_date - datetime.timedelta(days=i) for i in range(n_test_days)][::-1]

ls = LoadupSamples(
    train_start_date=global_start_date,
    test_dates=test_dates,
    treegroup=stock_group,
    timegroup=timegroup,
    params=params,
)

In [36]:
ls.load_samples(main_path = "../src/featureAlchemy/bin/")

Test date 2025-11-22 not found in the database. Omitting.
Test date 2025-11-23 not found in the database. Omitting.
Non-finite/too-large values in train y tree: 310 samples.
Non-finite/too-large values in train y time: 310 samples.
Removing 310 samples from training data.


In [37]:
Xtree = ls.train_Xtree
ytree = ls.train_ytree
Xtime = ls.train_Xtime
ytime = ls.train_ytime

treenames = ls.featureTreeNames
timenames = ls.featureTimeNames

meta_tr = ls.meta_pl_train
meta_te = ls.meta_pl_test

z= None
Z = None

In [38]:
np.mean(np.log(ytree[:, 4]))

np.float64(0.0018521554291240788)

In [39]:
eval = HelperMetrics.evaluate_mask_nullonempty(np.ones(Xtree.shape[0], dtype=bool), meta_tr.get_column("date"), ytree[:, 4])

print(f"Initial eval: {eval}")
print(f"Shape Xtree_close: {Xtree.shape}")
print(f"Shape ytree: {ytree.shape}")
print(f"Number of tickers: {meta_tr.get_column('ticker').n_unique()}")
print(f"Number of dates: {meta_tr.get_column('date').n_unique()}")

Initial eval: 1.0026616519991816
Shape Xtree_close: (143480, 1666)
Shape ytree: (143480, 5)
Number of tickers: 91
Number of dates: 1730


In [40]:
import re
tn = np.asarray(treenames, dtype=str)
mask_treenames = np.ones(len(treenames), dtype=bool)
max_lag = 10
if False:
    mask_selected |= np.char.find(tn, "FeatureTA_") >= 0
if True:
    mask_tn_lag = np.array(
        [(m:=re.search(r'_lag_m(\d+)', s)) is None or int(m.group(1)) <= max_lag
            for s in tn]
    )
    
mask_treenames &= mask_tn_lag
    
print(f"Using {np.sum(mask_treenames)} / {len(treenames)} tree features.")


Using 934 / 1666 tree features.


In [41]:
idx = []
idx.append(treenames.index("FeatureGroup_WeightedIndexMHPct_lag_m10_MH_4"))
idx.append(treenames.index("Seasonal_is_month_end"))
idx.append(treenames.index("FeatureTA_trend_mass_index_lag_m10"))
idx.append(treenames.index("FeatureGroup_RetGrLvl_lag_m1"))
idx.append(treenames.index("MathFeature_Price_logDiff"))
idx.append(treenames.index("MathFeature_Return_log"))

mask_treenames = np.zeros(len(treenames), dtype=bool)
mask_treenames[idx] = True

In [42]:
from src.hyperparameterTuning.StratSelectedMasks import StratSelectedMasks
precompute_params = {
    "atr_period"        : 5,   
    "atr_alpha"         : 0.171537,    
    "atr_qup"           : 0.848168,      
    "atr_qdown"         : 0.985,    
    "slope_period"      : 8,  
    "slope_qup"         : 0.632790,    
    "slope_qdown"       : 1.0, 
    "rmse_period"       : 32,  
    "rmse_delay"        : 4,   
    "rmse_ma_wndw"      : 17, 
    "rmse_alpha"        : 0.145176,   
    "rmse_qup"          : 0.732962,     
    "rmse_qdown"        : 0.987,   
    
    "sl0" : 0.823295, 
    "sl1" : 0.833562, 
    "sl2" : 0.783671, 
    "sl3" : 0.878303, 
    "sl4" : 0.982187, 

    "tp0" : 1.307256, 
    "tp1" : 1.442779, 
    "tp2" : 1.377516, 
    "tp3" : 1.211979, 
    "tp4" : 1.378628, 
}

mask_train, mask_test, _, _, _, _, _, _ = StratSelectedMasks().run(
    Xtree,
    Xtime,
    ytree,
    ls.train_ytree_low, 
    ls.train_ytree_high, 
    ls.train_ytree_open, 
    Xtree,
    Xtime,
    treenames,
    timenames,
    meta_tr,
    meta_tr,
    opt_params=precompute_params,
)

feats = [
    "MathFeature_Return_log",
    "MathFeature_Return_log_lag_m1",
]
mask_train = (Xtime[:, -1,0] < Xtime[:, -2,0]) & (Xtime[:, -2,0] < Xtime[:, -3,0])
mask_test = mask_train

  Precompute -> sl [0.823295 0.833562 0.783671 0.878303 0.982187] | tp: [1.307256 1.442779 1.377516 1.211979 1.378628]


## Best SL TP through Optuna

In [43]:
import optuna

ytree_masked = ls.train_ytree[mask_train]
ytree_low_masked = ls.train_ytree_low[mask_train]
ytree_high_masked = ls.train_ytree_high[mask_train]
ytree_open_masked = ls.train_ytree_open[mask_train]

def objective(trial):
    opt_params = {}
    opt_params["sl0"] = trial.suggest_float("sl0", 0.8, 0.94)
    opt_params["sl1"] = trial.suggest_float("sl1", 0.8, 0.99)
    opt_params["sl2"] = trial.suggest_float("sl2", 0.8, 1.05)
    opt_params["sl3"] = trial.suggest_float("sl3", 0.8, 1.1)
    opt_params["sl4"] = trial.suggest_float("sl4", 0.8, 1.2)
    
    opt_params["tp0"] = trial.suggest_float("tp0", 1.05, 1.6)
    opt_params["tp1"] = trial.suggest_float("tp1", 1.05, 1.8)
    opt_params["tp2"] = trial.suggest_float("tp2", 1.00, 1.8)
    opt_params["tp3"] = trial.suggest_float("tp3", 0.95, 1.9)
    opt_params["tp4"] = trial.suggest_float("tp4", 0.9, 1.95)

    sl_vec = [opt_params["sl0"], opt_params["sl1"], opt_params["sl2"], opt_params["sl3"], opt_params["sl4"]]
    tp_vec = [opt_params["tp0"], opt_params["tp1"], opt_params["tp2"], opt_params["tp3"], opt_params["tp4"]]
    sl_tr_mat, _, tp_tr_mat, _ = HelperSLTP.replicate(
        sl_vec,
        tp_vec,
        ytree_masked,
        ytree_masked,
    )
    
    res_vec = HelperMetrics.collapse_sl_tp(
        ytree_masked, 
        ytree_low_masked, 
        ytree_high_masked, 
        ytree_open_masked, 
        sl_tr_mat, 
        tp_tr_mat,
    )

    metric = HelperMetrics.evaluate_mask_oneonempty(
        np.ones(ytree_masked.shape[0], dtype=bool),
        meta_tr.get_column("date").filter(mask_train),
        res_vec,
    )
    score = metric
    trial.set_user_attr("score", score)

    return score

sampler = optuna.samplers.TPESampler(n_startup_trials=50)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=500)
df_atrtunnel: pd.DataFrame = study.trials_dataframe()
print("Best value:", study.best_value)
print("Best params:", study.best_params)

[I 2026-01-06 17:43:52,335] A new study created in memory with name: no-name-e34c4ba6-b2fe-4a81-b595-aeaa96c362fa
[I 2026-01-06 17:43:52,345] Trial 0 finished with value: 1.0026298994143124 and parameters: {'sl0': 0.8345164399040241, 'sl1': 0.8992150540143287, 'sl2': 0.8576657986882162, 'sl3': 1.0962029344775395, 'sl4': 0.9717953931106575, 'tp0': 1.1520770887945404, 'tp1': 1.762762079584753, 'tp2': 1.4212147505101527, 'tp3': 1.7551725830841116, 'tp4': 1.1163942176618715}. Best is trial 0 with value: 1.0026298994143124.
[I 2026-01-06 17:43:52,359] Trial 1 finished with value: 1.0018666388469153 and parameters: {'sl0': 0.8608793983756362, 'sl1': 0.9539927236055699, 'sl2': 1.0203407490395362, 'sl3': 0.8394952278556492, 'sl4': 1.1407158584736257, 'tp0': 1.2007943349526142, 'tp1': 1.7295952281479425, 'tp2': 1.4065738274221962, 'tp3': 1.8615802887424644, 'tp4': 1.435984636117606}. Best is trial 0 with value: 1.0026298994143124.
[I 2026-01-06 17:43:52,367] Trial 2 finished with value: 1.00331

Best value: 1.0040100974560733
Best params: {'sl0': 0.9033926266800509, 'sl1': 0.8638815528256732, 'sl2': 0.8786516517903566, 'sl3': 0.831640737199833, 'sl4': 1.0201246293731445, 'tp0': 1.3133527119031314, 'tp1': 1.3625547409024605, 'tp2': 1.4742172198578327, 'tp3': 1.3081373050486609, 'tp4': 1.4176146351020222}


In [44]:
for key, value in study.best_params.items():
    print(f"{key}: {value}")

sl0: 0.9033926266800509
sl1: 0.8638815528256732
sl2: 0.8786516517903566
sl3: 0.831640737199833
sl4: 1.0201246293731445
tp0: 1.3133527119031314
tp1: 1.3625547409024605
tp2: 1.4742172198578327
tp3: 1.3081373050486609
tp4: 1.4176146351020222


In [45]:
perfect_sl, perfect_tp = HelperSLTP.perfect_sl_tp(
    ytree_masked, 
    ytree_low_masked, 
    ytree_high_masked, 
    ytree_open_masked, 
    sl_max=2.0,
)

print("Perfect SL:", np.quantile(perfect_sl, np.linspace(0,1,11)))
print("Perfect TP:", np.quantile(perfect_tp, np.linspace(0,1,11)))

print(f"Perfect SL on lower than 0.99: {np.quantile(perfect_sl[perfect_sl < 0.99], np.linspace(0,1,11))}")
print(f"Perfect TP on higher than 1.01: {np.quantile(perfect_tp[perfect_tp > 1.01], np.linspace(0,1,11))}")

sl_tr_mat = np.repeat(perfect_sl[:, np.newaxis], 5, axis=1)
tp_tr_mat = np.repeat(perfect_tp[:, np.newaxis], 5, axis=1)
sl_tr_mat[:,1:] = 0.7
tp_tr_mat[:,1:] = 1.5
res_vec = HelperMetrics.collapse_sl_tp(
    ytree_masked, 
    ytree_low_masked, 
    ytree_high_masked, 
    ytree_open_masked, 
    sl_tr_mat, 
    tp_tr_mat,
)
metric = HelperMetrics.evaluate_mask_oneonempty(
    np.ones(ytree_masked.shape[0], dtype=bool),
    meta_tr.get_column("date").filter(mask_train),
    res_vec,
)
print(f"Stat SLTP result: {metric ** 52}")

Perfect SL: [0.8        0.95565891 0.97057491 0.97845178 0.98369895 0.9876168
 0.99079701 0.99341579 0.99565896 0.99783182 1.00496957]
Perfect TP: [1.005      1.005      1.00870476 1.01320525 1.01810753 1.02375844
 1.03031446 1.03842446 1.05012093 1.07158393 2.00034868]
Perfect SL on lower than 0.99: [0.8        0.94306835 0.95875466 0.96750505 0.97329187 0.9776563
 0.98106948 0.98379102 0.98610131 0.98820115 0.98999866]
Perfect TP on higher than 1.01: [1.01000046 1.0134837  1.01726868 1.02136525 1.0260951  1.03134341
 1.03759381 1.04581349 1.05796844 1.08034663 2.00034868]
Stat SLTP result: 1.803525268018982


In [46]:
stat_sl_tr_mat, _, stat_tp_tr_mat, _ = HelperSLTP.calc_conditional(
    ytree_masked, 
    ytree_low_masked, 
    ytree_high_masked, 
    ytree_open_masked, 
    nS_te=ytree_masked.shape[0],
    include_live_mask=False,
)

print("Stat SL:", stat_sl_tr_mat[0,:])
print("Stat TP:", stat_tp_tr_mat[0,:])

res_vec = HelperMetrics.collapse_sl_tp(
    ytree_masked, 
    ytree_low_masked, 
    ytree_high_masked, 
    ytree_open_masked, 
    stat_sl_tr_mat, 
    stat_tp_tr_mat,
)
metric = HelperMetrics.evaluate_mask_oneonempty(
    np.ones(ytree_masked.shape[0], dtype=bool),
    meta_tr.get_column("date").filter(mask_train),
    res_vec,
)
print(f"Stat SLTP result: {metric}")

Stat SL: [0.7        0.7        0.7        0.85100384 0.7       ]
Stat TP: [1.41520056 1.41520056 1.3264299  1.31001574 1.29713732]
Stat SLTP result: 1.0038394885269273


In [47]:
stat_sl_tr_mat, _, stat_tp_tr_mat, _ = HelperSLTP.calc_conditional(
    ytree_masked, 
    ytree_low_masked, 
    ytree_high_masked, 
    ytree_open_masked, 
    nS_te=ytree_masked.shape[0],
    include_live_mask=True,
)

print("Stat SL:", stat_sl_tr_mat[0,:])
print("Stat TP:", stat_tp_tr_mat[0,:])

res_vec = HelperMetrics.collapse_sl_tp(
    ytree_masked, 
    ytree_low_masked, 
    ytree_high_masked, 
    ytree_open_masked, 
    stat_sl_tr_mat, 
    stat_tp_tr_mat,
)
metric = HelperMetrics.evaluate_mask_oneonempty(
    np.ones(ytree_masked.shape[0], dtype=bool),
    meta_tr.get_column("date").filter(mask_train),
    res_vec,
)
print(f"Stat SLTP result: {metric}")

Stat SL: [0.7        0.72122905 0.73589758 0.85100384 0.7       ]
Stat TP: [1.41520056 1.21413721 1.33212341 1.31001574 1.29713732]
Stat SLTP result: 1.0038138108742274


In [63]:
feats = [
    "MathFeature_Return_log",
    "FeatureGroup_WeightedIndexPct",
    "FeatureTA_volatility_ui",
    "MathFeature_Drawdown_MH2",
    "MathFeature_PriceAdjustment",
    "FeatureGroup_WeightedIndexMHPct_lag_m1_MH_2",
    "FeatureGroup_AvgReturnPct_lag_m2",
    "FinData_quar_grossProfit_lagquot_qm5",
    "FeatureTA_volatility_bbli_lag_m2",
    "FeatureGroup_WeightedIndexMHPct_lag_m1_MH_6"
]
idx = [treenames.index(feats[6])]
Xtr_feats = Xtree[mask_train][:, idx]
Xte_feats = Xtree[mask_train][:, idx] # is not used
clust_sl_tr_mat, _, clust_tp_tr_mat, _ = HelperSLTP.sltp_clustered(
    ytree_masked, 
    ytree_low_masked, 
    ytree_high_masked, 
    ytree_open_masked, 
    Xtr_feats = Xtr_feats,
    Xte_feats = Xte_feats,
    n_clusters = 2000,
    include_live_mask=False,
)

print("Clust SL:", np.median(clust_sl_tr_mat, axis=0))
print("Clust TP:", np.median(clust_tp_tr_mat, axis=0))

res_vec = HelperMetrics.collapse_sl_tp(
    ytree_masked, 
    ytree_low_masked, 
    ytree_high_masked, 
    ytree_open_masked, 
    clust_sl_tr_mat, 
    clust_tp_tr_mat,
)
metric = HelperMetrics.evaluate_mask_oneonempty(
    np.ones(ytree_masked.shape[0], dtype=bool),
    meta_tr.get_column("date").filter(mask_train),
    res_vec,
)
print(f"Clust SLTP result: {metric}")

Clust SL: [0.98680279 0.98680279 0.98414693 0.98682709 0.98610018]
Clust TP: [1.01610738 1.01610738 1.02095745 1.02261445 1.02836469]
Clust SLTP result: 1.0061981079806606


In [58]:
print(mask_train.sum()/Xtree.shape[0])

0.22336214106495678


In [17]:
import numpy as np, polars as pl

name2i = {n:i for i,n in enumerate(treenames)}
def score(m):
    if isinstance(m,(int,float,np.floating)): return float(m)
    if isinstance(m,dict):
        for k in ("score","metric","value"): 
            if isinstance(m.get(k), (int,float,np.floating)): return float(m[k])
    return None

rows=[]
for f in treenames:
    i = name2i.get(f)
    print(f"Processing feature {f}, index {i} out of {len(treenames)}")
    try:
        X = Xtree[mask_train][:, [i]]
        k = min(1000, X.shape[0])
        sl,_,tp,_ = HelperSLTP.sltp_clustered(
            ytree_masked, ytree_low_masked, ytree_high_masked, ytree_open_masked,
            Xtr_feats=X, Xte_feats=X, n_clusters=k, include_live_mask=False
        )
        res = HelperMetrics.collapse_sl_tp(
            ytree_masked, ytree_low_masked, ytree_high_masked, ytree_open_masked, sl, tp
        )
        m = HelperMetrics.evaluate_mask_oneonempty(
            np.ones(ytree_masked.shape[0], bool),
            meta_tr.get_column("date").filter(mask_train),
            res
        )
        rows.append((f,i,True,None,k,float(np.median(sl,0)[0]),float(np.median(tp,0)[0]),m,score(m)))
    except Exception as e:
        rows.append((f,i,False,repr(e),None,None,None,None,None))

Processing feature Category_other, index 0 out of 1666
Processing feature Category_industrials, index 1 out of 1666
Processing feature Category_healthcare, index 2 out of 1666
Processing feature Category_technology, index 3 out of 1666
Processing feature Category_financial-services, index 4 out of 1666
Processing feature Category_real-estate, index 5 out of 1666
Processing feature Category_energy, index 6 out of 1666
Processing feature Category_consumer-cyclical, index 7 out of 1666
Processing feature Category_inSnP500, index 8 out of 1666
Processing feature Category_inNas100, index 9 out of 1666
Processing feature FinData_quar_grossProfit_nivRev, index 10 out of 1666
Processing feature FinData_quar_ebit_nivRev, index 11 out of 1666
Processing feature FinData_quar_ebitda_nivRev, index 12 out of 1666
Processing feature FinData_quar_totalAssets_nivRev, index 13 out of 1666
Processing feature FinData_quar_totalCurrentLiabilities_nivRev, index 14 out of 1666
Processing feature FinData_quar

KeyboardInterrupt: 

In [ ]:
df = pl.DataFrame(rows, schema=[
    "feature","feat_idx","ok","error","n_clusters","median_sl","median_tp","metric","score"
]).sort(pl.col("score"), descending=True)

df


C:\Users\kimer\AppData\Local\Temp\ipykernel_22760\987067248.py:1: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df = pl.DataFrame(rows, schema=[


feature,feat_idx,ok,error,n_clusters,median_sl,median_tp,metric,score
str,i64,bool,null,i64,f64,f64,f64,f64
"""FeatureGroup_WeightedIndexMHPc…",1653,true,null,1000,0.985981,1.01742,1.006118,1.006118
"""FeatureGroup_WeightedIndexMHPc…",1621,true,null,1000,0.98468,1.018433,1.006098,1.006098
"""FeatureGroup_AvgReturnPct_lag_…",1580,true,null,1000,0.984101,1.01783,1.006088,1.006088
"""FeatureGroup_WeightedIndexMHPc…",1624,true,null,1000,0.984101,1.016527,1.00602,1.00602
"""FeatureGroup_AvgReturnPct_lag_…",1550,true,null,1000,0.983875,1.017263,1.006019,1.006019
…,…,…,…,…,…,…,…,…
"""FeatureTA_momentum_stoch_lag_m…",920,true,null,1000,0.979914,1.023302,1.002882,1.002882
"""FeatureTA_momentum_wr_lag_m5""",922,true,null,1000,0.979914,1.023302,1.002879,1.002879
"""FeatureTA_trend_ema_slow_lag_m…",801,true,null,1000,0.982889,1.018657,1.002863,1.002863
